In [ ]:
import re

In [ ]:
import json

In [ ]:
import pickle

In [ ]:
import tqdm

In [ ]:
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
import numpy as np

In [ ]:
import pandas as pd

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
import seaborn as sns

In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM

In [ ]:
MODEL_ID = "google/gemma-4-E2B-it"

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID, temperature=0.1)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto"
)

In [ ]:
df = pd.read_excel("data.xlsx", header=0)

In [ ]:
df.head()

In [ ]:
df = df.fillna("")

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
with open("artificial_profiles.json", "r") as f:
    profiles = json.load(f)

In [ ]:
profiles.update({
'monetary_policy_researcher': 'A user, which is interesed in macroeconimcs in general and monetary police studies in particular',
'computer_vision_researcher': 'A user, which is interested in machine learning and computer vision in particular'
})

In [ ]:
profiles

In [ ]:
results = {}

In [ ]:
for profile in profiles:
    SYSTEM_PROMPT = f"""
    You are a {profile}.
    {profiles[profile]}
    Evaluate, how much you would like proposed student project based on its title in russian. Rate it on a scale [None, 1, 2, 3, 4, 5], where the higher the better.
    If proposed project title does not fit your interests, just return None.
    Return stricly JSON.
    For example.
    """
    SYSTEM_PROMPT += """
    ```json
    {'score': 4}
    ```
    ```json
    {'score': None}
    ```
    """

    messages = []
    scores = {}
    for row in df.iterrows():
        messages.append(
            (
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": row[1]["title_rus"]}
            )
        )
    for message in tqdm.tqdm(messages):
        text = processor.apply_chat_template(
            message, 
            tokenize=False, 
            add_generation_prompt=True, 
            enable_thinking=False
        )
        
        inputs = processor(text=text, return_tensors="pt").to(model.device)
        input_len = inputs["input_ids"].shape[-1]
        
        outputs = model.generate(**inputs, max_new_tokens=1024)
        response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

        try:
            json_match = re.search(r'```json\s*(.*?)\s*```', response, re.DOTALL)
            if json_match:
                json_str = json_match.group(1)
            else:
                json_str = response
            answer = json.loads(json_str)
        except Exception:
            answer = {"score": 'None'}
        
        score = answer.get("score", 'None')
        try:
            score = int(score)
        except Exception:
            score = None

        scores[message[1]["content"]] = score
    nans = [1 if x else 0 for x in list(scores.values())]
    if nans:
        nans_mean = np.round(np.mean(nans), 4)
    else:
        nans_mean = np.nan
    values = [x for x in list(scores.values()) if x]
    if values:
        values_median = np.median(values)
    else:
        values_median = np.nan
    print(f"{profile}: {nans_mean}, {values_median}")
    results[profile] = scores

In [ ]:
for researcher in results:
    fig, ax = plt.subplots(figsize=(10, 6))
    scores_list = list(results[researcher].values())
    
    sns.histplot(scores_list, ax=ax, alpha=0.7, kde=True)
    
    ax.set_title(f'Rating Distribution - {researcher}', fontsize=14, fontweight='bold')
    ax.set_xlabel('Score', fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    
    unique_scores = sorted(set([x for x in scores_list if x is not None]))
    if unique_scores:
        ax.legend([f'Scores: {unique_scores}'], title='Score Values', loc='best')
    
    plt.tight_layout()
    plt.show()

In [ ]:
for researcher in results:
    data = [x for x in results[researcher].items() if x[1]]
    if data:
        for topic, score in sorted(data, key=lambda x: x[1], reverse=True)[:10]:
            print(f"{researcher}, {topic}, {score}")
    print()

In [ ]:
with open("artificial_profiles_scores.pkl", "wb") as f:
    pickle.dump(results, f)